In [79]:
import requests
import datetime
import json
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import mean_absolute_error

In [2]:
URL_PRIVAT24 = "https://api.privatbank.ua/p24api/exchange_rates?date={date}"

In [3]:
#01.12.2015
URL_PRIVAT24.format(date="01.12.2015")

'https://api.privatbank.ua/p24api/exchange_rates?date=01.12.2015'

In [4]:
base = datetime.datetime.today()

In [5]:
base

datetime.datetime(2026, 2, 19, 13, 20, 7, 821440)

In [6]:
dir(base)

['__add__',
 '__class__',
 '__delattr__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__ne__',
 '__new__',
 '__radd__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__rsub__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__sub__',
 '__subclasshook__',
 'astimezone',
 'combine',
 'ctime',
 'date',
 'day',
 'dst',
 'fold',
 'fromisocalendar',
 'fromisoformat',
 'fromordinal',
 'fromtimestamp',
 'hour',
 'isocalendar',
 'isoformat',
 'isoweekday',
 'max',
 'microsecond',
 'min',
 'minute',
 'month',
 'now',
 'replace',
 'resolution',
 'second',
 'strftime',
 'strptime',
 'time',
 'timestamp',
 'timetuple',
 'timetz',
 'today',
 'toordinal',
 'tzinfo',
 'tzname',
 'utcfromtimestamp',
 'utcnow',
 'utcoffset',
 'utctimetuple',
 'weekday',
 'year']

In [7]:
from datetime import datetime, timedelta
now = datetime.now()
print(now.strftime("%d.%m.%Y"))

19.02.2026


In [8]:
num_days = 100

In [9]:
data = []
for x in range(num_days):
    item= base - timedelta(days=x)
    strochka = item.strftime("%d.%m.%Y")
    url = URL_PRIVAT24.format(date= strochka)
    print(url)
    r = requests.get(url)
    r.raise_for_status()
    vova=json.loads(r.content)
    data.append(vova)

https://api.privatbank.ua/p24api/exchange_rates?date=19.02.2026
https://api.privatbank.ua/p24api/exchange_rates?date=18.02.2026
https://api.privatbank.ua/p24api/exchange_rates?date=17.02.2026
https://api.privatbank.ua/p24api/exchange_rates?date=16.02.2026
https://api.privatbank.ua/p24api/exchange_rates?date=15.02.2026
https://api.privatbank.ua/p24api/exchange_rates?date=14.02.2026
https://api.privatbank.ua/p24api/exchange_rates?date=13.02.2026
https://api.privatbank.ua/p24api/exchange_rates?date=12.02.2026
https://api.privatbank.ua/p24api/exchange_rates?date=11.02.2026
https://api.privatbank.ua/p24api/exchange_rates?date=10.02.2026
https://api.privatbank.ua/p24api/exchange_rates?date=09.02.2026
https://api.privatbank.ua/p24api/exchange_rates?date=08.02.2026
https://api.privatbank.ua/p24api/exchange_rates?date=07.02.2026
https://api.privatbank.ua/p24api/exchange_rates?date=06.02.2026
https://api.privatbank.ua/p24api/exchange_rates?date=05.02.2026
https://api.privatbank.ua/p24api/exchang

In [10]:
r.raise_for_status()

In [11]:
# dir(r)

In [12]:
json.loads(r.content)

{'date': '12.11.2025',
 'bank': 'PB',
 'baseCurrency': 980,
 'baseCurrencyLit': 'UAH',
 'exchangeRate': [{'baseCurrency': 'UAH',
   'currency': 'AUD',
   'saleRateNB': 27.4057,
   'purchaseRateNB': 27.4057},
  {'baseCurrency': 'UAH',
   'currency': 'AZN',
   'saleRateNB': 24.7141,
   'purchaseRateNB': 24.7141},
  {'baseCurrency': 'UAH',
   'currency': 'BYN',
   'saleRateNB': 15.0766,
   'purchaseRateNB': 15.0766},
  {'baseCurrency': 'UAH',
   'currency': 'CAD',
   'saleRateNB': 29.9479,
   'purchaseRateNB': 29.9479},
  {'baseCurrency': 'UAH',
   'currency': 'CHF',
   'saleRateNB': 52.3929,
   'purchaseRateNB': 52.3929,
   'saleRate': 53.9,
   'purchaseRate': 52.35},
  {'baseCurrency': 'UAH',
   'currency': 'CNY',
   'saleRateNB': 5.902,
   'purchaseRateNB': 5.902},
  {'baseCurrency': 'UAH',
   'currency': 'CZK',
   'saleRateNB': 2.0017,
   'purchaseRateNB': 2.0017,
   'saleRate': 1.991,
   'purchaseRate': 1.6},
  {'baseCurrency': 'UAH',
   'currency': 'DKK',
   'saleRateNB': 6.5093,
  

In [13]:
def filter_ua_usd(item):
    return item["currency"] == "USD"
list(filter(filter_ua_usd, json.loads(r.content)["exchangeRate"]))[0]

{'baseCurrency': 'UAH',
 'currency': 'USD',
 'saleRateNB': 42.0139,
 'purchaseRateNB': 42.0139,
 'saleRate': 42.17,
 'purchaseRate': 41.57}

In [14]:
missing_count = sum(1 for item in data if not list(filter(filter_ua_usd, item["exchangeRate"])))
print(f"Кількість елементів без курсу: {missing_count} з {len(data)}")

Кількість елементів без курсу: 0 з 100


In [15]:
window = 10
ua_list = []

for idx, element in enumerate(data):
    # знаходимо курс в поточному елементі
    current_rate = list(filter(filter_ua_usd, element["exchangeRate"]))

    if current_rate:   # беремо якщо знайшли
        rate_value = current_rate[0]["saleRateNB"]
    else: # якщо не знайшлося треба інтерполювати
        # беремо попереднє значення з ua_list
        previous_rate = ua_list[-1] if ua_list else None

        # шукаємо наступне значення
        next_rate = None
        for j in range(idx + 1, len(data)):
            temp = list(filter(filter_ua_usd, data[j]["exchangeRate"]))
            if temp:
                next_rate = temp[0]["saleRateNB"]
                break

        # рахування середнього
        if previous_rate and next_rate:
            rate_value = (previous_rate + next_rate) / 2  # інтерполяція
        elif previous_rate:
            rate_value = previous_rate # якщо наступного немає беремо попереднє
        elif next_rate:
            rate_value = next_rate # якщо попереднього немає беремо наступнеЯЯя
        else:
            rate_value = 0 # якщо взагалі нічого немає не має статися

    # додаємо результат до списку
    ua_list.append(rate_value)

In [42]:
ua_list = ua_list[::-1]

In [43]:
windows_size = 10
x = []
y = []
iter = range(len(ua_list) -windows_size)

In [44]:
for i in iter:
  x.append(ua_list[i:i +window])
  y.append(ua_list[window +i])


In [45]:
x = np.array(x)
y = np.array(y)

In [46]:
stop = len(x) * 0.8

In [47]:
stop = int(stop)
x_train = x[:stop]
x_test = x[stop:]
y_train = y[:stop]
y_test = y[stop:]

In [56]:
x_train = torch.tensor(x_train, dtype = torch.float32)
x_test = torch.tensor(x_test, dtype = torch.float32)
y_train = torch.tensor(y_train, dtype = torch.float32)
y_test = torch.tensor(y_test, dtype = torch.float32)

/tmp/ipython-input-3533140091.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x_train = torch.tensor(x_train, dtype = torch.float32)
/tmp/ipython-input-3533140091.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x_test = torch.tensor(x_test, dtype = torch.float32)
/tmp/ipython-input-3533140091.py:3: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_train = torch.tensor(y_train, dtype = torch.float32)
/tmp/ipython-input-3533140091.py:4: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone(

In [57]:
rnn = nn.RNN(input_size = 1, hidden_size = 64, batch_first = True)

In [58]:
lin = nn.Linear(64, out_features = 1)

In [67]:
optimizer = torch.optim.Adam(lr = 0.001, params = list(rnn.parameters())+list(lin.parameters()))

In [82]:
epoch = 100
b_s = 32

In [83]:
loss_fn = nn.MSELoss()

In [84]:
for eph in range(epoch):
  rnn.train()
  lin.train()
  for r in range(0,len(x_train), b_s):
    x_b = x_train[r:r+b_s]
    y_b = y_train[r:r+b_s]
    optimizer.zero_grad()
    x_b = x_b.unsqueeze(-1)
    y_b = y_b.unsqueeze(-1)
    out,_ = rnn(x_b)
    prediction = lin(out[:,-1,:])
    loss = loss_fn(prediction, y_b)
    loss.backward()
    optimizer.step()
    print(loss)


tensor(1637.6899, grad_fn=<MseLossBackward0>)
tensor(1668.0502, grad_fn=<MseLossBackward0>)
tensor(1680.0997, grad_fn=<MseLossBackward0>)
tensor(1613.3672, grad_fn=<MseLossBackward0>)
tensor(1643.9541, grad_fn=<MseLossBackward0>)
tensor(1656.2753, grad_fn=<MseLossBackward0>)
tensor(1590.2618, grad_fn=<MseLossBackward0>)
tensor(1620.9377, grad_fn=<MseLossBackward0>)
tensor(1633.4922, grad_fn=<MseLossBackward0>)
tensor(1568.2729, grad_fn=<MseLossBackward0>)
tensor(1599.1501, grad_fn=<MseLossBackward0>)
tensor(1612.0476, grad_fn=<MseLossBackward0>)
tensor(1547.6346, grad_fn=<MseLossBackward0>)
tensor(1578.6732, grad_fn=<MseLossBackward0>)
tensor(1591.7761, grad_fn=<MseLossBackward0>)
tensor(1527.9167, grad_fn=<MseLossBackward0>)
tensor(1558.9259, grad_fn=<MseLossBackward0>)
tensor(1572.0564, grad_fn=<MseLossBackward0>)
tensor(1508.6062, grad_fn=<MseLossBackward0>)
tensor(1539.4935, grad_fn=<MseLossBackward0>)
tensor(1552.5817, grad_fn=<MseLossBackward0>)
tensor(1489.4886, grad_fn=<MseLoss

In [85]:
_

tensor([[[-0.9998, -1.0000,  1.0000,  0.9999,  0.9998,  1.0000, -0.9999,
          -0.9997,  1.0000,  0.9996, -0.9993, -1.0000,  1.0000,  0.9996,
          -0.9993,  1.0000,  1.0000,  0.9995,  1.0000, -1.0000, -0.9998,
          -1.0000,  0.9999, -0.9997, -1.0000, -0.9994, -0.9993, -0.9996,
           1.0000,  0.9998, -0.9998, -0.9995,  0.9997, -0.9999,  1.0000,
           0.9995, -0.9990,  1.0000, -1.0000, -1.0000,  0.9998, -1.0000,
          -1.0000,  0.9997,  0.9999,  0.9992, -1.0000,  0.9999, -0.9999,
           0.9999,  0.9991,  0.9995, -1.0000,  0.9999, -0.9993,  1.0000,
          -0.9996,  0.9999, -0.9997,  0.9999, -1.0000, -1.0000,  0.9998,
          -0.9994],
         [-0.9998, -1.0000,  1.0000,  0.9999,  0.9998,  1.0000, -0.9999,
          -0.9997,  1.0000,  0.9996, -0.9993, -1.0000,  1.0000,  0.9996,
          -0.9993,  1.0000,  1.0000,  0.9995,  1.0000, -1.0000, -0.9998,
          -1.0000,  0.9999, -0.9997, -1.0000, -0.9994, -0.9993, -0.9996,
           1.0000,  0.9998, -0.

In [86]:
out.shape

torch.Size([8, 10, 64])

In [87]:
out[:,-1,:].shape

torch.Size([8, 64])

In [88]:
rnn.eval()
lin.eval()

Linear(in_features=64, out_features=1, bias=True)

In [89]:
with torch.no_grad():
  x_test_y = x_test.unsqueeze(-1)
  out,_ = rnn(x_test_y)
  prediction = lin(out[:,-1,:])


In [90]:
mean_absolute_error(y_test, prediction.numpy())

23.006893157958984